In [1]:
import pandas as pd
import numpy as np

def select_stratified_samples(input_file, output_file, total_samples=150):
    # 1. 讀取資料集
    df = pd.read_csv(input_file)
    
    # 2. 定義目標類別與每個難度要抽取的數量
    target_categories = ['ORDER', 'REFUND', 'PAYMENT', 'SHIPPING', 'DELIVERY']
    difficulty_levels = [1, 2, 3]
    samples_per_diff = total_samples // len(difficulty_levels) # 50 筆
    
    final_list = []

    for diff in difficulty_levels:
        # 篩選該難度的資料
        df_diff = df[df['difficulty_level'] == diff]
        
        # 找出該難度中存在的目標類別
        available_cats = [c for c in target_categories if c in df_diff['category'].unique()]
        
        if not available_cats:
            # 如果該難度完全沒有目標類別，則隨機抽取
            diff_samples = df_diff.sample(n=samples_per_diff, random_state=42)
        else:
            # 計算每個類別應分配的筆數 (例如 50 / 5 = 10)
            num_per_cat = samples_per_diff // len(available_cats)
            
            diff_selection = []
            for cat in available_cats:
                cat_df = df_diff[df_diff['category'] == cat]
                # 確保不會抽超過現有的資料量
                pick_count = min(len(cat_df), num_per_cat)
                diff_selection.append(cat_df.sample(n=pick_count, random_state=42))
            
            # 合併已抽取的類別資料
            current_df = pd.concat(diff_selection)
            
            # 如果因為除法餘數或資料不足，導致不到 50 筆，則從該難度剩下的池子隨機補足
            if len(current_df) < samples_per_diff:
                needed = samples_per_diff - len(current_df)
                remaining_pool = df_diff.drop(current_df.index)
                extra_samples = remaining_pool.sample(n=needed, random_state=42)
                current_df = pd.concat([current_df, extra_samples])
            elif len(current_df) > samples_per_diff:
                # 如果超過 50 筆，隨機修剪
                current_df = current_df.sample(n=samples_per_diff, random_state=42)
        
        final_list.append(current_df)

    # 3. 合併結果並輸出
    final_df = pd.concat(final_list)
    final_df.to_csv(output_file, index=False)
    
    print(f"✅ 成功挑選 {len(final_df)} 筆資料，已儲存至: {output_file}")
    
    # 顯示統計結果確認公平性
    print("\n--- 抽樣分佈統計 ---")
    print(pd.crosstab(final_df['difficulty_level'], final_df['category']))

if __name__ == "__main__":
    # 執行函數
    select_stratified_samples('bitext_with_difficulty.csv', 'selected_test_samples_150.csv')

✅ 成功挑選 150 筆資料，已儲存至: selected_test_samples_150.csv

--- 抽樣分佈統計 ---
category          ACCOUNT  DELIVERY  INVOICE  ORDER  PAYMENT  REFUND  SHIPPING
difficulty_level                                                              
1                       1        10        1     10       10      11         7
2                       0        10        0     10       10      10        10
3                       0        10        0     10       10      10        10


In [1]:
import pandas as pd
import json
import random
import string

def generate_random_id(length=8):
    """生成隨機編號，如 ORD12345"""
    digits = ''.join(random.choices(string.digits, k=length))
    return f"ORD{digits}"

def create_fact_sheets(input_csv, output_json):
    df = pd.read_csv(input_csv)
    fact_sheets = {}

    # 設定隨機種子確保實驗可重複性
    random.seed(42)

    for index, row in df.iterrows():
        case_id = f"CASE_{index+1:03d}"
        category = row['category']
        
        # 模擬這筆資料的「真實情況」
        order_number = generate_random_id()
        
        # 基礎事實架構
        fact_data = {
            "metadata": {
                "original_index": index,
                "category": category,
                "intent": row['intent'],
                "difficulty": int(row['difficulty_level'])
            },
            "ground_truth": {
                "customer_info": {
                    "name": random.choice(["Alex Chen", "Jamie Wu", "Taylor Lin", "Jordan Wang"]),
                    "email": f"user{index+1}@example.com",
                    "phone": f"0912-345-{index+1:03d}"
                },
                "order_info": {
                    "order_number": order_number,
                    "status": random.choice(["Shipped", "Processing", "Delivered", "Refunded"]),
                    "amount": round(random.uniform(50, 500), 2),
                    "currency": "USD",
                    "items": [f"Product_{random.randint(1, 100)}"],
                    "shipping_date": "2024-05-01"
                },
                "refund_info": {
                    "refund_status": "Pending" if category == "REFUND" else "N/A",
                    "refund_amount": round(random.uniform(50, 500), 2) if category == "REFUND" else 0
                }
            }
        }
        
        # 將生成的隨機資料填回指令中，讓 Agent 有對象可以查
        # 這裡會替換 instruction 裡的 {{Order Number}}
        modified_instruction = row['instruction'].replace("{{Order Number}}", order_number)
        fact_data["agent_input"] = modified_instruction
        
        fact_sheets[case_id] = fact_data

    # 儲存為單一 JSON 檔案
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(fact_sheets, f, indent=4, ensure_ascii=False)

    print(f"✅ Fact Sheets 已生成，共計 {len(fact_sheets)} 筆案例。")
    print(f"📂 儲存路徑: {output_json}")

if __name__ == "__main__":
    create_fact_sheets('selected_test_samples_150.csv', 'fact_sheets.json')

✅ Fact Sheets 已生成，共計 150 筆案例。
📂 儲存路徑: fact_sheets.json
